# 2026 Global and regional hackathon

* For the 2026 hackathon: https://digital-earths-uk-hackathon.github.io/
* This notebook will get you started with the regional, Africa LAM 4.4-km RAL3.3, nested in the N1280 CoMA9 global model.
* There are equivalent LAMs for South Americe (SAmer) and Southeast Asia (SEA)
* There are also LAMS with different physics, using the trailblazer variant of CoMoroph (CoMA9_TBv1).

## Open dataset from catalog

In [ ]:
import cartopy.crs as ccrs
import intake
import matplotlib.pyplot as plt

import pandas as pd

import easygems.healpix as egh

from utils import hp_mods, plot_all_fields

In [ ]:
# Filter out annoying warning.
import warnings
warnings.filterwarnings("ignore", message=".*The return type of `Dataset.dims` will be changed.*", category=FutureWarning)

In [ ]:
# Open catalog.
url = 'https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml'
cat = intake.open_catalog(url)['UK']
# Use online if not on JASMIN.
# cat = intake.open_catalog('https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml')['online']

In [ ]:
# Load Africa LAM.
sim = 'um_Africa_km4p4_RAL3P3_n1280_CoMA9_nest_hk26'
sim_cat = cat[sim]

In [ ]:
# Open a 1h (2D) and 3h (3D) dataset.
ds1h = sim_cat(zoom=8, time='PT1H').to_dask().pipe(hp_mods)
ds3h = sim_cat(zoom=8, time='PT3H').to_dask().pipe(hp_mods)

## Display data

In [ ]:
# How can I plot the data?
# What is 'tas'?
egh.healpix_show(ds1h.tas.isel(time=13))

In [ ]:
# Let's see all the hourly fields at the second timestamp (00:00 has some variables without data).
# .compute() loads all of the data from the object store into RAM.
ds1h_plot = ds1h.sel(time=pd.Timestamp('2020-01-20 01:00')).compute()

In [ ]:
# Plot all fields!
plot_all_fields(ds1h_plot)

In [ ]:
# 3-hourly data (3D data) at 900 hPa.
ds3h_plot = ds3h.sel(time=pd.Timestamp('2020-01-20 03:00')).sel(pressure=900).compute()

In [ ]:
plot_all_fields(ds3h_plot)

In [ ]:
# Check that there's more cloud liquid water in the lower troposphere than stratosphere.
ds3h.isel(time=1).clw.mean(dim='cell').plot(y='pressure', yincrease=False)

## Look at timeseries data

In [ ]:
# We can load the data at different zooms. What are the time chunks for zoom 0? Why is this useful?
ds1h_z0 = sim_cat(zoom=0, time='PT1H').to_dask().pipe(hp_mods)
ds1h_z0

In [ ]:
ds1h_z0.pr.mean(dim='cell').plot()